# 🛒 Projet Final - Segmentation Clients E-commerce

## 🎯 Objectif
Segmenter les clients d'une boutique en ligne pour personnaliser les stratégies marketing.

## 📊 Contexte
Vous travaillez pour une entreprise e-commerce qui souhaite mieux comprendre ses clients. Votre mission : identifier des groupes de clients avec des comportements similaires pour adapter les campagnes marketing.

## 📋 Dataset
- **clients_ecommerce.csv** : 800 clients avec leurs comportements d'achat
- **Features** : âge, ancienneté, nombre d'achats, montants, engagement, etc.



---
## 📦 Étape 1 : Import des bibliothèques

**📝 À faire :** Importer toutes les bibliothèques nécessaires

In [ ]:
# Manipulation de données
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Clustering
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage

# Métriques
from sklearn.metrics import silhouette_score

# Configuration
plt.style.use('ggplot')
sns.set_palette('Set2')
%matplotlib inline

print("✅ Bibliothèques importées!")

---
## 📂 Étape 2 : Chargement des données

**📝 À faire :** Charger le dataset et afficher les premières lignes

In [ ]:
# Charger les données
df = pd.read_csv('data/clients_ecommerce.csv')

print(f"📊 Dataset chargé : {df.shape[0]} clients, {df.shape[1]} colonnes")
print("\n📋 Aperçu des données:")
df.head(10)

---
## 🔍 Étape 3 : Exploration des données (EDA)

**📝 À faire :** Comprendre les données avant de clustériser

In [ ]:
# Informations générales
print("📊 Informations du dataset:")
print(df.info())

print("\n📈 Statistiques descriptives:")
df.describe()

In [ ]:
# Vérifier les valeurs manquantes
print("❓ Valeurs manquantes par colonne:")
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Nombre': missing,
    'Pourcentage': missing_pct
})
print(missing_df[missing_df['Nombre'] > 0])

In [ ]:
# Visualisation des distributions
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

for idx, col in enumerate(numeric_cols):
    if idx < len(axes):
        axes[idx].hist(df[col].dropna(), bins=30, edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'Distribution: {col}', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Fréquence')
        axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Distributions visualisées!")

### 💡 Question d'analyse

**❓ Quelles variables semblent avoir des outliers ?**

*(Répondre ici en markdown)*

Réponse : 

---
## 🧹 Étape 4 : Préparation des données

**📝 À faire :** Nettoyer et préparer les données pour le clustering

### 4.1 - Gestion des valeurs manquantes

**💡 Indice :** Utiliser la médiane pour les valeurs numériques

In [ ]:
# SOLUTION FOURNIE : Imputation par la médiane
df_clean = df.copy()

for col in df_clean.select_dtypes(include=[np.number]).columns:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f"✅ {col}: {df[col].isnull().sum()} valeurs manquantes remplacées par {median_val:.2f}")

print(f"\n✅ Total valeurs manquantes après imputation : {df_clean.isnull().sum().sum()}")

### 4.2 - Détection et traitement des outliers

**💡 Méthode :** Z-score > 3 ou IQR

In [ ]:
# Visualisation des outliers avec boxplots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

important_cols = ['nb_achats', 'montant_total', 'panier_moyen', 
                  'jours_depuis_dernier_achat', 'nb_pages_vues', 'taux_retour']

for idx, col in enumerate(important_cols):
    axes[idx].boxplot(df_clean[col].dropna(), vert=True)
    axes[idx].set_title(f'Boxplot: {col}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel(col)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# SOLUTION FOURNIE : Détection outliers par Z-score
from scipy import stats

df_no_outliers = df_clean.copy()
outlier_indices = set()

for col in ['montant_total', 'panier_moyen', 'nb_pages_vues']:
    z_scores = np.abs(stats.zscore(df_no_outliers[col]))
    outliers = np.where(z_scores > 3)[0]
    outlier_indices.update(outliers)
    print(f"📊 {col}: {len(outliers)} outliers détectés (Z-score > 3)")

print(f"\n🎯 Total indices uniques avec outliers: {len(outlier_indices)}")

# Option 1: Supprimer les outliers (décommenter si souhaité)
# df_no_outliers = df_no_outliers.drop(list(outlier_indices)).reset_index(drop=True)

# Option 2: Winsorization (cap aux percentiles) - RECOMMANDÉ
for col in ['montant_total', 'panier_moyen', 'nb_pages_vues']:
    p99 = df_no_outliers[col].quantile(0.99)
    df_no_outliers[col] = df_no_outliers[col].clip(upper=p99)
    
print(f"✅ Outliers traités par winsorization (cap au 99e percentile)")
print(f"📊 Dataset final: {df_no_outliers.shape[0]} clients")

### 4.3 - Sélection des features pour le clustering

**💡 Important :** Ne pas inclure client_id dans le clustering!

In [ ]:
# Sélectionner les features pertinentes (RFM + engagement)
features_clustering = [
    'nb_achats',                    # Fréquence (F)
    'montant_total',                # Montant (M)
    'jours_depuis_dernier_achat',   # Récence (R)
    'panier_moyen',                 # Valeur moyenne
    'nb_pages_vues',                # Engagement
    'taux_conversion',              # Qualité engagement
    'taux_retour'                   # Satisfaction
]

X = df_no_outliers[features_clustering].copy()

print(f"✅ Features sélectionnées pour clustering:")
for feat in features_clustering:
    print(f"   - {feat}")
    
print(f"\n📊 Shape des données: {X.shape}")

### 4.4 - Normalisation des données

**⚠️ Important :** Les algorithmes de clustering sont sensibles aux échelles!

In [ ]:
# SOLUTION FOURNIE : Standardisation (Z-score normalization)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✅ Données normalisées (moyenne=0, écart-type=1)")
print(f"\n📊 Vérification après normalisation:")
print(f"   - Moyenne par feature: {X_scaled.mean(axis=0).round(2)}")
print(f"   - Écart-type par feature: {X_scaled.std(axis=0).round(2)}")

---
## 🎯 Étape 5 : Déterminer le nombre optimal de clusters (K)

**📝 À faire :** Utiliser la méthode du coude (Elbow Method)

In [ ]:
# SOLUTION FOURNIE : Elbow Method
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Graphique 1: Inertie (Elbow)
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Nombre de clusters (K)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Inertie (WCSS)', fontsize=12, fontweight='bold')
axes[0].set_title('Méthode du Coude', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(K_range)

# Graphique 2: Silhouette Score
axes[1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Nombre de clusters (K)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Silhouette Score', fontsize=12, fontweight='bold')
axes[1].set_title('Score de Silhouette', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(K_range)

plt.tight_layout()
plt.show()

print("✅ Graphiques générés!")
print(f"\n📊 Meilleur Silhouette Score: K={K_range[np.argmax(silhouette_scores)]} (score={max(silhouette_scores):.3f})")

### 💡 Question d'analyse

**❓ Quel est le nombre optimal de clusters selon :**
- Méthode du coude : K = ?
- Silhouette score : K = ?

*(Répondre ici en markdown)*

Réponse : 

---
## 🔬 Étape 6 : Appliquer K-Means

**📝 À faire :** Appliquer K-Means avec le K optimal choisi

In [ ]:
# SOLUTION FOURNIE : K-Means avec K=4 (à ajuster selon votre analyse)
K_optimal = 4

kmeans_final = KMeans(n_clusters=K_optimal, random_state=42, n_init=10)
clusters = kmeans_final.fit_predict(X_scaled)

# Ajouter les clusters au dataframe
df_no_outliers['cluster'] = clusters

print(f"✅ K-Means appliqué avec K={K_optimal}")
print(f"\n📊 Distribution des clusters:")
print(df_no_outliers['cluster'].value_counts().sort_index())

# Silhouette score final
silhouette_final = silhouette_score(X_scaled, clusters)
print(f"\n🎯 Silhouette Score final: {silhouette_final:.3f}")

---
## 📊 Étape 7 : Analyser et interpréter les clusters

**📝 À faire :** Comprendre les caractéristiques de chaque segment

In [ ]:
# Statistiques par cluster
cluster_analysis = df_no_outliers.groupby('cluster')[features_clustering].mean()

print("📊 Profil moyen de chaque cluster:\n")
print(cluster_analysis.round(2))

In [ ]:
# Visualisation des profils
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, col in enumerate(features_clustering):
    if idx < len(axes):
        df_no_outliers.boxplot(column=col, by='cluster', ax=axes[idx])
        axes[idx].set_title(f'{col} par cluster', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Cluster')
        axes[idx].set_ylabel(col)

plt.suptitle('Comparaison des Clusters', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap des moyennes par cluster
plt.figure(figsize=(12, 6))

# Normaliser pour la visualisation
cluster_analysis_norm = (cluster_analysis - cluster_analysis.min()) / (cluster_analysis.max() - cluster_analysis.min())

sns.heatmap(cluster_analysis_norm.T, annot=True, fmt='.2f', cmap='RdYlGn', 
            cbar_kws={'label': 'Valeur normalisée (0-1)'}, linewidths=0.5)
plt.title('Heatmap des Profils de Clusters (normalisé)', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Cluster', fontsize=12, fontweight='bold')
plt.ylabel('Features', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Heatmap des clusters générée!")

### 💡 Interprétation des clusters

**📝 À compléter :** Donner un nom et une description à chaque cluster

#### Cluster 0 :
- **Nom suggéré :** _________
- **Caractéristiques :** _________
- **Stratégie marketing :** _________

#### Cluster 1 :
- **Nom suggéré :** _________
- **Caractéristiques :** _________
- **Stratégie marketing :** _________

#### Cluster 2 :
- **Nom suggéré :** _________
- **Caractéristiques :** _________
- **Stratégie marketing :** _________

#### Cluster 3 :
- **Nom suggéré :** _________
- **Caractéristiques :** _________
- **Stratégie marketing :** _________

---
## 🔄 Étape 8 : Comparer avec d'autres méthodes (Bonus)

**📝 À faire :** Tester Clustering Hiérarchique et DBSCAN

### 8.1 - Clustering Hiérarchique

In [ ]:
# SOLUTION FOURNIE : Dendrogramme
# Prendre un échantillon pour la visualisation (dendrogramme complet serait illisible)
sample_size = 100
sample_indices = np.random.choice(len(X_scaled), sample_size, replace=False)
X_sample = X_scaled[sample_indices]

# Calculer le linkage
linkage_matrix = linkage(X_sample, method='ward')

# Dendrogramme
plt.figure(figsize=(16, 8))
dendrogram(linkage_matrix, truncate_mode='level', p=5)
plt.title('Dendrogramme - Clustering Hiérarchique (échantillon de 100 clients)', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Index Client (ou taille du cluster)', fontsize=12)
plt.ylabel('Distance', fontsize=12)
plt.axhline(y=20, color='red', linestyle='--', linewidth=2, label='Seuil de coupure suggéré')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Dendrogramme généré!")

### 8.2 - DBSCAN

In [ ]:
# SOLUTION FOURNIE : DBSCAN
# Tester différentes valeurs d'eps
dbscan = DBSCAN(eps=2.5, min_samples=10)
dbscan_clusters = dbscan.fit_predict(X_scaled)

n_clusters_dbscan = len(set(dbscan_clusters)) - (1 if -1 in dbscan_clusters else 0)
n_noise = list(dbscan_clusters).count(-1)

print(f"✅ DBSCAN appliqué (eps=2.5, min_samples=10)")
print(f"📊 Nombre de clusters trouvés: {n_clusters_dbscan}")
print(f"📊 Nombre de points bruit (outliers): {n_noise}")
print(f"\n📊 Distribution:")
unique, counts = np.unique(dbscan_clusters, return_counts=True)
for cluster_id, count in zip(unique, counts):
    label = 'Bruit' if cluster_id == -1 else f'Cluster {cluster_id}'
    print(f"   {label}: {count} clients ({count/len(dbscan_clusters)*100:.1f}%)")

---
## 📈 Étape 9 : Visualisation en 2D (PCA)

**📝 À faire :** Réduire les dimensions pour visualiser les clusters

In [ ]:
# SOLUTION FOURNIE : PCA pour visualisation
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"✅ PCA appliquée: {X_scaled.shape[1]} dimensions → 2 dimensions")
print(f"📊 Variance expliquée: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# Visualisation des clusters K-Means en 2D
plt.figure(figsize=(14, 8))

scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='Set2', 
                     s=50, alpha=0.6, edgecolor='black', linewidth=0.5)

# Centres des clusters en 2D
centers_pca = pca.transform(kmeans_final.cluster_centers_)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], c='red', s=300, 
           marker='*', edgecolor='black', linewidth=2, label='Centres', zorder=10)

plt.title(f'Visualisation des Clusters K-Means (K={K_optimal}) - PCA 2D', 
         fontsize=16, fontweight='bold', pad=20)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
plt.colorbar(scatter, label='Cluster')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Visualisation 2D générée!")

---
## 💾 Étape 10 : Sauvegarder les résultats

**📝 À faire :** Exporter les segments pour utilisation marketing

In [ ]:
# Créer un mapping de noms pour les clusters
cluster_names = {
    0: 'Segment_0',  # À personnaliser selon votre analyse
    1: 'Segment_1',
    2: 'Segment_2',
    3: 'Segment_3'
}

df_final = df_no_outliers.copy()
df_final['segment_nom'] = df_final['cluster'].map(cluster_names)

# Sauvegarder
df_final.to_csv('data/clients_segments.csv', index=False)

print("✅ Résultats sauvegardés dans: data/clients_segments.csv")
print(f"\n📊 Aperçu du fichier final:")
df_final[['client_id', 'nb_achats', 'montant_total', 'cluster', 'segment_nom']].head(10)

---
## 📝 Étape 11 : Rapport final

**📝 À compléter :** Synthèse du projet

### 🎯 Synthèse des résultats

#### 1. Méthodologie
- **Algorithme principal :** K-Means avec K=___
- **Features utilisées :** ___ (RFM + engagement)
- **Prétraitement :** Imputation médiane, traitement outliers, normalisation
- **Qualité :** Silhouette Score = ___

#### 2. Segments identifiés
*(Décrire brièvement chaque segment)*

#### 3. Recommandations business
*(Actions marketing par segment)*

#### 4. Limites et améliorations possibles
*(Points d'attention et pistes d'optimisation)*

---
## 🎉 Projet terminé!

### ✅ Ce que vous avez appris:
1. ✅ Exploration et nettoyage de données e-commerce
2. ✅ Détection et traitement des outliers
3. ✅ Normalisation pour le clustering
4. ✅ Méthode du coude pour choisir K
5. ✅ Application de K-Means
6. ✅ Interprétation business des clusters
7. ✅ Comparaison avec autres méthodes (Hiérarchique, DBSCAN)
8. ✅ Visualisation PCA

### 🚀 Pour aller plus loin:
- Tester d'autres features combinaisons
- Utiliser le clustering pour prédiction (classification supervisée)
- Automatiser la segmentation avec un pipeline
- Intégrer dans un système de recommandation